## WikipediaRetriever

In [1]:
from langchain_community.retrievers import WikipediaRetriever


In [2]:
#initilaise the retriever( optional: set language and top_k)

retriever= WikipediaRetriever(top_k_results=2,lang='en')

In [3]:
#Define your query
query='the geopolitical history of india and pakistan from the perspective of a chinese'

#get relevent wikipedia Documents
docs= retriever.invoke(query)

In [4]:
#print retrieved content
for i, doc in enumerate(docs):
    print(f"-------Result {i+1}---")
    print(f"Content:\n{doc.page_content}....")

-------Result 1---
Content:
The United States has been providing military aid and economic assistance to Pakistan for various purposes since 1948. In 2017, the U.S. stopped military aid to Pakistan, which was about US$2 billion per year. With U.S. military assistance suspended in 2018 and civilian aid reduced to about $300 million for 2022, Pakistani authorities have turned to other countries for help.


== History ==
From 1947 to 1958, under civilian leadership, the United States provided Pakistan with modest economic aid and limited military assistance. During this period, Pakistan became a member of the South East Asian Treaty Organization (SEATO) and the Central Treaty Organization (CENTO), after a Mutual Defence Assistance Agreement signed in May 1954, which facilitated increased levels of both economic and military aid from the U.S.
In 1958, Ayub Khan led Pakistan's first military coup, becoming Chief Martial Law Administrator (CMLA) and later President until 1969. During his ten

## Vector Store Retriever

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

/home/section/Desktop/Files/MTECH IIT/Git projects/Projects/Langchain-basics/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [7]:
#Step 2 Initialise embedding model
embedding_model= OpenAIEmbeddings()

#Step3 Create Chroma vector space in memory
vector_store=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name='mycollection'
)

In [8]:
#Step4 Convert Vector stores into reterivers

retriver=vector_store.as_retriever(search_kwargs={'k':2})

In [9]:
query="What is chroma used for?"

results= retriever.invoke(query)

In [10]:
#print retrieved content
for i, doc in enumerate(results):
    print(f"-------Result {i+1}---")
    print(f"Content:\n{doc.page_content}....")

-------Result 1---
Content:
Chroma key compositing, or chroma keying, is a visual-effects and post-production technique for compositing (layering) two or more images or video streams together based on colour hues (chroma range). The technique has been used in many fields to remove a background from the subject of a photo or video — particularly the newscasting, motion picture, and video game industries. A colour range in the foreground footage is made transparent, allowing separately filmed background footage or a static image to be inserted into the scene. The chroma keying technique is commonly used in video production and post-production. This technique is also referred to as colour keying, colour separation overlay (CSO; primarily by the BBC), or by various terms for specific colour-related variants such as green screen or blue screen; chroma keying can be done with backgrounds of any colour that are uniform and distinct, but green and blue backgrounds are more commonly used becaus

## MMR

In [11]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [16]:
from langchain_community.vectorstores import FAISS

#Initialise OpenAI Embeddings
embedding_model= OpenAIEmbeddings()

#Step 2: Create the FIASS vector store

vector_store= FAISS.from_documents(

    documents=docs,
    embedding=embedding_model
)

In [24]:
#Enable MMR in the retriever
retriver =vector_store.as_retriever(
    search_type='mmr',  #<---this enables mmr
    search_kwargs={"k":3,"lambda_mult":0.5}  #k= top results,lambda_mult= relevenace-diversity balance
)

In [25]:
query="what is lang chain"
reults= retriver.invoke(query)

In [26]:
#print retrieved content
for i, doc in enumerate(reults):
    print(f"-------Result {i+1}---")
    print(f"Content:\n{doc.page_content}....")

-------Result 1---
Content:
LangChain is used to build LLM based applications.....
-------Result 2---
Content:
Chroma is used to store and search document embeddings.....
-------Result 3---
Content:
LangChain supports Chroma, FAISS, Pinecone, and more.....


# multi Query Retriever

In [28]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import MultiQueryRetriever

In [36]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [37]:
#Initialise OPEN AI Embeddings
embedding_model= OpenAIEmbeddings()

In [38]:
vector_store= FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

In [39]:
#create retrievers
similarity_retriever= vector_store.as_retriever(search_type='similarity',search_kwargs={"k":5})

In [40]:
multi_query_retriver=MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k":5}),
    llm=ChatOpenAI()
)

In [41]:
query= "how to improve energy levels and maintain balance?"

#Retriev results
similarity_result=similarity_retriever.invoke(query)
multi_query_result=multi_query_retriver.invoke(query)

In [42]:
#print retrieved content
for i, doc in enumerate(similarity_result):
    print(f"-------Result {i+1}---")
    print(f"Content:\n{doc.page_content}....")

print("*"*100)

#print retrieved content
for i, doc in enumerate(multi_query_result):
    print(f"-------Result {i+1}---")
    print(f"Content:\n{doc.page_content}....")

-------Result 1---
Content:
Drinking sufficient water throughout the day helps maintain metabolism and energy.....
-------Result 2---
Content:
Mindfulness and controlled breathing lower cortisol and improve mental clarity.....
-------Result 3---
Content:
Regular walking boosts heart health and can reduce symptoms of depression.....
-------Result 4---
Content:
Deep sleep is crucial for cellular repair and emotional regulation.....
-------Result 5---
Content:
The solar energy system in modern homes helps balance electricity demand.....
****************************************************************************************************
-------Result 1---
Content:
Drinking sufficient water throughout the day helps maintain metabolism and energy.....
-------Result 2---
Content:
Mindfulness and controlled breathing lower cortisol and improve mental clarity.....
-------Result 3---
Content:
Deep sleep is crucial for cellular repair and emotional regulation.....
-------Result 4---
Content:
Regu

## Contextual Compression Retriever

In [46]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [47]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [50]:
embedding_model=OpenAIEmbeddings()
#Create FIASS vector store and Embedding model
vector_store= FAISS.from_documents(docs,embedding=embedding_model)

In [51]:
base_retriver=vector_store.as_retriever(search_kwargs={"k":5})

In [52]:
#Set up the compressor using LLM
llm=ChatOpenAI()
compressor= LLMChainExtractor.from_llm(llm)



In [ ]:
#Create the contextual compression retriver
compression_retriver=ContextualCompressionRetriever(
    base_retriever=base_retriver,
    
)